## 1. Tool(도구)의 정의와 역할

### 1.1 ReAct에서 Tool이란 무엇인가

ReAct 프레임워크에서 **Tool(도구)** 은 LLM이 외부 세계와 상호작용하기 위해 호출하는 실행 가능한 함수 또는 서비스를 의미한다. LLM은 자연어를 이해하고 생성하는 능력이 뛰어나지만, 다음과 같은 한계를 가진다.

- **수학 연산**: 복잡한 계산에서 오류를 범할 수 있다
- **최신 정보**: 학습 데이터 이후의 정보를 알 수 없다
- **외부 시스템 접근**: 데이터베이스, 파일 시스템, 웹 검색 등에 직접 접근할 수 없다
- **정확한 데이터 조회**: 정확한 수치나 사실 정보를 보장할 수 없다

Tool은 이러한 한계를 보완하여, LLM의 추론 능력과 외부 도구의 실행 능력을 결합하는 핵심 요소이다.

### 1.2 Tool의 필수 요소

모든 Tool은 다음 5가지 요소를 갖추어야 한다.

| 요소 | 설명 | 예시 |
|------|------|------|
| **이름(Name)** | 도구를 식별하는 고유한 이름 | `Calculator`, `WebSearch` |
| **설명(Description)** | 도구의 기능을 LLM이 이해할 수 있도록 기술 | "수학 수식을 계산합니다" |
| **입력 스키마(Input Schema)** | 도구가 받는 입력의 형식과 제약 조건 | `{"type": "string"}` |
| **실행 로직(Execute Logic)** | 입력을 받아 실제 작업을 수행하는 코드 | `eval(expression)` |
| **출력 형식(Output Format)** | 결과를 문자열로 반환하는 규칙 | `"계산 결과: 42"` |

### 1.3 Tool이 LLM의 한계를 보완하는 방식

ReAct에서 LLM과 Tool의 협업은 다음과 같은 흐름을 따른다.

```
LLM(Thought) -> 어떤 도구를 사용할지 결정
LLM(Action)  -> 도구 이름과 입력값을 지정
Tool(실행)    -> 실제 연산/조회 수행
LLM(관찰)    -> 도구 결과를 받아 다음 추론에 활용
```

이 구조에서 LLM은 **"무엇을 해야 하는가"**를 결정하는 두뇌 역할을 하고, Tool은 **"실제로 수행"**하는 손과 발 역할을 한다.

## 2. Tool 스키마 설계

### 2.1 JSON Schema 기반 Tool 정의

도구를 체계적으로 정의하기 위해 JSON Schema 형식을 사용한다. 이는 OpenAI의 Function Calling이나 LangChain 등 주요 프레임워크에서도 채택하고 있는 표준 방식이다.

```json
{
  "name": "Calculator",
  "description": "수학 수식을 계산합니다.",
  "parameters": {
    "type": "string",
    "description": "계산할 수학 수식 (예: 2 + 3 * 4)"
  }
}
```

### 2.2 매개변수 타입과 필수/선택 지정

복잡한 도구의 경우, 매개변수를 세부적으로 정의할 수 있다.

```json
{
  "name": "WebSearch",
  "description": "웹에서 정보를 검색합니다.",
  "parameters": {
    "type": "object",
    "properties": {
      "query": {"type": "string", "description": "검색 질의"},
      "max_results": {"type": "integer", "description": "최대 결과 수", "default": 5}
    },
    "required": ["query"]
  }
}
```

### 2.3 Tool 설명문 작성 가이드라인

도구의 설명문은 LLM이 적절한 도구를 선택하는 데 결정적인 역할을 한다. 다음 원칙을 따르는 것이 좋다.

1. **명확성**: 도구가 무엇을 하는지 한 문장으로 설명한다
2. **입력 예시**: 어떤 형식의 입력을 기대하는지 예시를 제공한다
3. **제한 사항**: 도구가 처리할 수 없는 경우를 명시한다
4. **사용 시점**: 언제 이 도구를 사용해야 하는지 안내한다

####  기본 Tool 클래스 설계

이제 파이썬의 추상 클래스(ABC)를 활용하여 Tool의 기본 구조를 설계하고, 세 가지 구체적인 도구를 구현한다.

- **CalculatorTool**: 수학 수식을 평가하는 계산기
- **StringProcessorTool**: 문자열 통계를 분석하는 도구
- **DictionaryTool**: 미리 정의된 사전에서 용어를 검색하는 도구

In [18]:
# abc  (Abstract Base Class)  - 추상클래스
from abc import ABC, abstractmethod
from typing import Any, Dict,Optional
import json
class BaseTool(ABC):  # 추상클래스
    '''React 에이전트에서 사용하는 도구의 기본 클래스'''
    def __init__(self,name:str, description:str):
        self.name = name; self.description = description
    @abstractmethod
    def execute(self, input_text:str) ->str:
        '''도구를 실행하고 결과를 문자열로 반환'''
        pass

    def get_schema(self)->Dict:
        '''도구의 스키마를 반환'''
        return{
            'name' : self.name,
            'description':self.description,
            'parameters':self._get_parameters()
        }
    def _get_parameters(self)->str:
        return {'type':'string','description':'입력테스트'}
    def __repr__(self):  # 객체를 출력할때
        return f'Tool(name={self.name})'   


In [19]:
class Person:
    def __init__(self,name):
        self.name = name
p = Person("hong gil dong")        
print(p)

In [20]:
class Person:
    def __init__(self,name):
        self.name = name
    def __repr__(self):
        return f'ok'
p = Person("hong gil dong")        
print(p)

ok


In [21]:
class CalculatorTool(BaseTool):
    def __init__(self):
        super().__init__(
            name='Calculator',
            description='수학 수식을 계산합니다. 사칙연산 거듭제곱 나머지연산 등을 지원합니다.'
        )
    def execute(self, input_text:str) ->str:
        try:
            # 안전한 수식 평가를 위해 허용된 연산자만 사용
            allowed_chars = set('0123456789+-*/.() ')
            if not all(c in allowed_chars for c in input_text):
                return f"오류: 허용되지 않는 문자가 포함되어 있습니다."
            result = eval(input_text)
            return f"계산 결과: {result}"
        except Exception as e:
            return f"계산 오류: {str(e)}"
class StringProcessorTool(BaseTool):
    """문자열 처리 도구"""
    
    def __init__(self):
        super().__init__(
            name="StringProcessor",
            description="문자열의 길이, 단어 수, 글자 수 등을 분석합니다."
        )
    
    def execute(self, input_text: str) -> str:
        char_count = len(input_text)
        word_count = len(input_text.split())
        line_count = len(input_text.splitlines())
        return (f"문자 수: {char_count}, 단어 수: {word_count}, "
                f"줄 수: {line_count}")

class DictionaryTool(BaseTool):
    """미리 정의된 사전에서 정보를 검색하는 도구"""
    
    def __init__(self):
        super().__init__(
            name="Dictionary",
            description="내장 사전에서 용어의 정의를 검색합니다."
        )
        self.entries = {
            "인공지능": "인간의 학습, 추론, 지각 능력을 컴퓨터로 구현하는 기술",
            "머신러닝": "데이터로부터 패턴을 학습하여 예측이나 결정을 수행하는 AI의 하위 분야",
            "딥러닝": "다층 신경망을 사용하여 복잡한 패턴을 학습하는 머신러닝의 하위 분야",
            "자연어처리": "컴퓨터가 인간의 언어를 이해하고 생성하는 AI 기술",
            "강화학습": "환경과의 상호작용을 통해 보상을 최대화하는 행동을 학습하는 방법",
            "트랜스포머": "어텐션 메커니즘 기반의 신경망 아키텍처로, 현대 NLP의 기반",
            "LLM": "대규모 텍스트 데이터로 사전 학습된 거대 언어 모델",
            "프롬프트": "LLM에 입력되는 지시문 또는 질의문",
            "ReAct": "추론(Reasoning)과 행동(Acting)을 결합한 프롬프팅 기법"
        }
    
    def execute(self, input_text: str) -> str:
        term = input_text.strip()
        if term in self.entries:
            return f"{term}: {self.entries[term]}"
        # Partial match
        matches = [k for k in self.entries if term in k or k in term]
        if matches:
            results = [f"{k}: {self.entries[k]}" for k in matches]
            return "관련 항목:\n" + "\n".join(results)
        return f"'{term}'에 대한 정보를 찾을 수 없습니다."        



In [22]:
# Test Tool
calc = CalculatorTool()
print(f'schema : {json.dumps(calc.get_schema(),ensure_ascii=False,indent=3 )}')
print(f'실행 : {calc.execute("(15+25)*3")}')
print(f'오류 : {calc.execute("import os")}')

string_proc = StringProcessorTool()
print(f'실행:{string_proc.execute("ReAct는 대세입니다.")}')

dict_tool = DictionaryTool()
print(f'실행:{dict_tool.execute("ReAct는 대세입니다. 트랜스포머")}')

schema : {
   "name": "Calculator",
   "description": "수학 수식을 계산합니다. 사칙연산 거듭제곱 나머지연산 등을 지원합니다.",
   "parameters": {
      "type": "string",
      "description": "입력테스트"
   }
}
실행 : 계산 결과: 120
오류 : 오류: 허용되지 않는 문자가 포함되어 있습니다.
실행:문자 수: 13, 단어 수: 2, 줄 수: 1
실행:관련 항목:
트랜스포머: 어텐션 메커니즘 기반의 신경망 아키텍처로, 현대 NLP의 기반
ReAct: 추론(Reasoning)과 행동(Acting)을 결합한 프롬프팅 기법


## 3. Tool Registry 패턴

### 3.1 Registry의 개념과 필요성

실제 ReAct 에이전트는 여러 도구를 동시에 관리해야 한다. 이때 **Registry(레지스트리)** 패턴을 사용하면 다음과 같은 이점이 있다.

1. **중앙 집중 관리**: 모든 도구를 한 곳에서 등록하고 조회할 수 있다
2. **동적 확장**: 런타임에 새로운 도구를 추가하거나 제거할 수 있다
3. **일관된 인터페이스**: 도구 이름만으로 실행할 수 있는 통일된 API를 제공한다
4. **오류 처리**: 존재하지 않는 도구 호출 등의 예외를 중앙에서 처리한다

### 3.2 도구 등록/조회/실행 흐름

```
1. register(tool)  -> 도구를 이름으로 등록
2. get(name)       -> 이름으로 도구 객체 조회
3. execute(name, input) -> 이름과 입력으로 도구 실행
4. list_tools()    -> 등록된 모든 도구 목록 반환
```

이 흐름은 ReAct 루프에서 LLM이 `Action: ToolName[input]` 형식으로 도구를 호출할 때 그대로 적용된다.

In [23]:
class ToolRegistry:
    '''도구를 등록하고 관리하는 레지스트리'''
    def __init__(self):
        self._tools:Dict[str,BaseTool] = {}
    def register(self, tool:BaseTool) -> 'ToolRegistry':
        self._tools[tool.name] = tool        
        return self
    def get(self,name:str) -> Optional[BaseTool]:
        return self._tools.get(name)
    def execute(self, tool_name:str, input_text:str)->str:
        tool = self.get(tool_name)
        if tool is None:
            return f'오류 : {tool_name} 도구를 찾을수 없습니다. 사용가능한 도구 : {self._tools.keys()}'
        try:
            return tool.execute(input_text)
        except Exception as e:
            return f'실행 오류 : {e}'
    def list_tools(self)->str:
        descriptions = []
        for name, tool in self._tools.items():
            descriptions.append(f'  - {name} : {tool.description}')
        return "사용가능한도구\n" + "\n".join(descriptions)
    def get_all_schemas(self)->list:
        return [ tool.get_schema() for tool in self._tools.values()]
    
# Create and populate registry
print("=== Tool Registry 구성 ===")
registry = ToolRegistry()
registry.register(CalculatorTool())
registry.register(StringProcessorTool())
registry.register(DictionaryTool())

print(f"\n{registry.list_tools()}")

print("\n=== Registry를 통한 도구 실행 ===")
print(registry.execute("Calculator", "2 ** 10"))
print(registry.execute("Dictionary", "딥러닝"))
print(registry.execute("UnknownTool", "test"))    

=== Tool Registry 구성 ===

사용가능한도구
  - Calculator : 수학 수식을 계산합니다. 사칙연산 거듭제곱 나머지연산 등을 지원합니다.
  - StringProcessor : 문자열의 길이, 단어 수, 글자 수 등을 분석합니다.
  - Dictionary : 내장 사전에서 용어의 정의를 검색합니다.

=== Registry를 통한 도구 실행 ===
계산 결과: 1024
딥러닝: 다층 신경망을 사용하여 복잡한 패턴을 학습하는 머신러닝의 하위 분야
오류 : UnknownTool 도구를 찾을수 없습니다. 사용가능한 도구 : dict_keys(['Calculator', 'StringProcessor', 'Dictionary'])


## 4. Tool 체이닝

### 4.1 체이닝의 개념

**Tool 체이닝(Tool Chaining)** 이란 하나의 도구 실행 결과를 다른 도구의 입력으로 사용하는 패턴을 말한다. 복잡한 문제를 해결할 때, 단일 도구만으로는 충분하지 않은 경우가 많다.

예를 들어 다음과 같은 질문을 생각해보자.

> "한국 GDP를 인구수로 나누면 1인당 GDP는 얼마인가?"

이 질문에 답하려면:
1. 지식베이스에서 한국 GDP 조회 (KnowledgeBase 도구)
2. 지식베이스에서 한국 인구수 조회 (KnowledgeBase 도구)
3. GDP / 인구수 계산 (Calculator 도구)

이처럼 여러 도구를 순차적으로 호출하는 것이 체이닝이다.

### 4.2 ReAct에서의 체이닝

ReAct 프레임워크에서는 LLM이 자연스럽게 체이닝을 수행한다. 매 단계마다 이전 Observation을 참고하여 다음 Action을 결정하기 때문에, 개발자가 체이닝 로직을 명시적으로 구현할 필요가 없다. LLM 자체가 체이닝의 "오케스트레이터" 역할을 한다.

In [ ]:
def tool_chain(registry, steps):
    """여러 도구를 순차적으로 실행하는 체이닝 함수
    
    Args:
        registry: ToolRegistry 인스턴스
        steps: (tool_name, input_data) 리스트
               input_data가 callable이면 이전 결과를 인자로 받음
    """
    print("=== Tool 체이닝 실행 ===")
    previous_result = None
    
    for i, (tool_name, input_data) in enumerate(steps, 1):
        # input_data가 callable이면 이전 결과를 사용하여 입력 생성
        if callable(input_data):
            actual_input = input_data(previous_result)
        else:
            actual_input = input_data
        
        print(f"\n[Chain Step {i}] {tool_name}")
        print(f"  입력: {actual_input}")
        result = registry.execute(tool_name, actual_input)
        print(f"  출력: {result}")
        previous_result = result
    
    print(f"\n최종 결과: {previous_result}")
    return previous_result

# 체이닝 예시 1: 계산 결과 -> 문자열 분석
print("[예시 1] 계산 결과를 문자열로 분석")
chain_steps = [
    ("Calculator", "(100 + 200 + 300) / 3"),
    ("StringProcessor", lambda prev: prev),  # 이전 결과를 문자열로 분석
]
tool_chain(registry, chain_steps)

# 체이닝 예시 2: 사전 검색 -> 결과 분석 -> 계산
print("\n" + "-" * 40)
print("\n[예시 2] 사전 검색 -> 결과 분석 -> 계산")
chain_steps_2 = [
    ("Dictionary", "LLM"),
    ("StringProcessor", lambda prev: prev),
    ("Calculator", "7 * 8 + 6"),
]
tool_chain(registry, chain_steps_2)